# Overview — Student Performance Dataset

**Fonte:** UCI Machine Learning Repository — ID 320  
**Objectivo:** Perceber com o que estamos a trabalhar antes de injectar missingness artificial.

O dataset recolhe informação demográfica, social e escolar de alunos do ensino secundário em Portugal (disciplina de Matemática).  
A variável alvo é **G1** — nota do primeiro período (escala 0–20).

---

**Estrutura do notebook**

| Secção | O que faz |
|---|---|
| **1. Carregar** | Carrega o dataset raw (sem encoding) para manter os valores legíveis |
| **2. Tipos e nulos** | Tabela com dtype, n_unique, missing% e exemplo de cada variável |
| **3. Estatísticas descritivas** | `describe()` para numéricas + frequências detalhadas para categóricas |
| **4. G1 – distribuição** | Histograma + boxplot; imprime média, mediana, % aprovados |
| **5. Numéricas – distribuições** | Barchart de frequências para todas as ordinais; histograma para `absences` e `age` |
| **6. Categóricas – distribuições** | Grid de barcharts para todas as binárias/nominais |
| **7. Relação com G1** | Correlação de Pearson com barras coloridas; scatterplots das top features; boxplot por `failures` |
| **8. Matriz de correlação** | Heatmap completo + lista automática de pares com \|r\| > 0.4 |
| **9. Álcool e faltas** | Boxplots de G1 por Dalc/Walc; correlação álcool–faltas |
| **10. Faltas vs G1** | Scatter colorido por `studytime` com hover interactivo |
| **11. Educação dos pais** | Média de G1 por nível de Medu e Fedu com labels legíveis |
| **12. Conclusões** | Tabela com os achados mais importantes para apresentar aos tutores |

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ucimlrepo import fetch_ucirepo

## 1. Carregar o dataset

Carregamos o dataset **sem encoding** para que os valores se mantenham legíveis durante a exploração.

In [ ]:
student_performance = fetch_ucirepo(id=320)
X_raw = student_performance.data.features.copy()
y     = student_performance.data.targets.iloc[:, 0].rename("G1")
df    = pd.concat([X_raw, y], axis=1)

print(f"Dimensão: {df.shape[0]} observações × {df.shape[1]} variáveis")
df.head(10)

## 2. Tipos de variáveis e valores nulos

In [ ]:
info = pd.DataFrame({
    "dtype":     df.dtypes,
    "n_unique":  df.nunique(),
    "missing":   df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "exemplo":   df.iloc[0]
})
info

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols.remove("G1")  # separar o target

print(f"Variáveis categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Variáveis numéricas   ({len(num_cols)}): {num_cols}")
print(f"Target: G1")

## 3. Estatísticas descritivas

In [ ]:
df[num_cols + ["G1"]].describe().T.round(2)

In [ ]:
for col in cat_cols:
    vc  = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    print(f"\n{col}:")
    for v, c, p in zip(vc.index, vc.values, pct.values):
        print(f"  {str(v):<15} {c:>4}  ({p}%)")

## 4. Variável alvo — G1 (nota do 1.º período)

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuição de G1", "Boxplot de G1"])

fig.add_trace(go.Histogram(x=df["G1"], nbinsx=20,
                           marker_color="steelblue", name="G1"), row=1, col=1)
fig.add_trace(go.Box(y=df["G1"], marker_color="steelblue",
                     name="G1", boxmean=True), row=1, col=2)

fig.update_layout(title_text="Distribuição da nota G1", showlegend=False)
fig.update_xaxes(title_text="G1", row=1, col=1)
fig.update_yaxes(title_text="Contagem", row=1, col=1)
fig.show()

aprovados = (df["G1"] >= 10).mean() * 100
print(f"Média:    {df['G1'].mean():.2f}")
print(f"Mediana:  {df['G1'].median():.1f}")
print(f"Std:      {df['G1'].std():.2f}")
print(f"Min/Max:  {df['G1'].min()} / {df['G1'].max()}")
print(f"Nota 0:   {(df['G1']==0).sum()} alunos  ({(df['G1']==0).mean()*100:.1f}%)")
print(f"≥ 10:     {(df['G1']>=10).sum()} alunos  ({aprovados:.1f}%)")

## 5. Numéricas — distribuições

Barchart de frequências para variáveis ordinais; histograma para `absences` e `age`.

In [ ]:
ordinal_vars   = ["Medu", "Fedu", "traveltime", "studytime", "failures",
                  "famrel", "freetime", "goout", "Dalc", "Walc", "health"]
continuous_vars = ["age", "absences"]

cols_per_row = 4
n_rows = int(np.ceil(len(ordinal_vars) / cols_per_row))
fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=ordinal_vars)

for i, col in enumerate(ordinal_vars):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="steelblue", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Frequências das variáveis ordinais",
                  height=300 * n_rows)
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuição de absences", "Distribuição de age"])
fig.add_trace(go.Histogram(x=df["absences"], nbinsx=30,
                           marker_color="coral", name="absences"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["age"], nbinsx=8,
                           marker_color="mediumseagreen", name="age"), row=1, col=2)
fig.update_layout(showlegend=False)
fig.show()

print(f"absences — Mediana: {df['absences'].median():.0f}  "
      f"Max: {df['absences'].max()}  "
      f"Alunos com 0 faltas: {(df['absences']==0).sum()} "
      f"({(df['absences']==0).mean()*100:.1f}%)")

## 6. Categóricas — distribuições

Grid de barcharts para todas as variáveis binárias e nominais.

In [ ]:
cols_per_row = 3
n_rows = int(np.ceil(len(cat_cols) / cols_per_row))
fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=cat_cols)

for i, col in enumerate(cat_cols):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    vc = df[col].value_counts()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="mediumpurple", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Frequências das variáveis categóricas",
                  height=350 * n_rows)
fig.show()

## 7. Relação das features com G1

In [ ]:
# Correlação de Pearson com G1 (só numéricas)
corr_with_g1 = df[num_cols + ["G1"]].corr()["G1"].drop("G1").sort_values()

fig = px.bar(x=corr_with_g1.values, y=corr_with_g1.index,
             orientation="h",
             color=corr_with_g1.values,
             color_continuous_scale="RdBu",
             color_continuous_midpoint=0,
             title="Correlação de Pearson com G1",
             labels={"x": "Correlação", "y": "Variável"})
fig.update_layout(coloraxis_showscale=False)
fig.show()

print("Top 3 correlações positivas com G1:")
print(corr_with_g1.tail(3).to_string())
print("\nTop 3 correlações negativas com G1:")
print(corr_with_g1.head(3).to_string())

In [ ]:
# Scatterplots das variáveis mais correlacionadas com G1
key_vars = ["failures", "absences", "studytime", "Medu"]
fig = make_subplots(rows=1, cols=len(key_vars),
                    subplot_titles=[f"{v} vs G1" for v in key_vars])

rng = np.random.default_rng(42)
for i, v in enumerate(key_vars):
    fig.add_trace(go.Scatter(
        x=df[v] + rng.uniform(-0.15, 0.15, len(df)),
        y=df["G1"] + rng.uniform(-0.15, 0.15, len(df)),
        mode="markers",
        marker=dict(size=5, opacity=0.5, color="steelblue"),
        showlegend=False
    ), row=1, col=i+1)
    fig.update_xaxes(title_text=v, row=1, col=i+1)

fig.update_yaxes(title_text="G1", row=1, col=1)
fig.update_layout(title_text="Variáveis mais relevantes vs G1", height=400)
fig.show()

In [ ]:
# G1 por número de reprovações anteriores — variável com maior correlação negativa
fig = px.box(df, x="failures", y="G1", color="failures",
             title="G1 por número de reprovações anteriores (failures)",
             labels={"failures": "Reprovações anteriores", "G1": "Nota G1"},
             color_discrete_sequence=px.colors.sequential.Reds_r)
fig.update_layout(showlegend=False)
fig.show()

print(df.groupby("failures")["G1"].agg(["mean", "median", "count"]).round(2))

## 8. Matriz de correlação entre variáveis numéricas

In [ ]:
corr = df[num_cols + ["G1"]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale="RdBu",
    zmid=0,
    text=corr.values,
    texttemplate="%{text}",
    colorbar=dict(title="r")
))
fig.update_layout(title="Matriz de correlação (Pearson) — variáveis numéricas",
                  width=750, height=700)
fig.show()

# Pares com correlação alta (|r| > 0.4), excluindo diagonal
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack().reset_index()
)
corr_pairs.columns = ["var1", "var2", "r"]
strong = corr_pairs[corr_pairs["r"].abs() > 0.4].sort_values("r", ascending=False)
print("Pares com |r| > 0.4:")
print(strong.to_string(index=False))

## 9. Álcool e faltas

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Consumo diário (Dalc) vs G1",
                                    "Consumo semanal (Walc) vs G1"])
for col, c in [("Dalc", 1), ("Walc", 2)]:
    for name, group in df.groupby(col)["G1"]:
        fig.add_trace(go.Box(y=group, name=str(name),
                             marker_color=f"rgba(70,130,180,{0.4+name*0.1})",
                             showlegend=False), row=1, col=c)

fig.update_xaxes(title_text="Nível de consumo", row=1, col=1)
fig.update_xaxes(title_text="Nível de consumo", row=1, col=2)
fig.update_layout(title_text="Consumo de álcool vs nota G1")
fig.show()

print(f"Correlação Dalc–absences:  {df[['Dalc', 'absences']].corr().iloc[0,1]:.3f}")
print(f"Correlação Walc–absences:  {df[['Walc', 'absences']].corr().iloc[0,1]:.3f}")

## 10. Faltas vs G1

In [ ]:
fig = px.scatter(df, x="absences", y="G1",
                 color="studytime",
                 size="studytime",
                 hover_data=["failures", "Dalc", "sex"],
                 color_continuous_scale="Blues",
                 title="Faltas vs G1 — colorido por tempo de estudo",
                 labels={"absences": "Faltas", "G1": "Nota G1",
                         "studytime": "Estudo (h/sem)"})
fig.show()

## 11. Educação dos pais vs G1

In [ ]:
edu_labels = {0: "Nenhuma", 1: "4.º ano", 2: "9.º ano",
              3: "Secundário", 4: "Superior"}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Educação da mãe (Medu) vs G1",
                                    "Educação do pai (Fedu) vs G1"])
for col, c in [("Medu", 1), ("Fedu", 2)]:
    groups = df.groupby(col)["G1"].mean().reset_index()
    groups["label"] = groups[col].map(edu_labels)
    fig.add_trace(go.Bar(
        x=groups["label"], y=groups["G1"].round(2),
        text=groups["G1"].round(2), textposition="outside",
        marker_color="mediumseagreen", showlegend=False
    ), row=1, col=c)
    fig.update_yaxes(title_text="Média G1", row=1, col=c)

fig.update_layout(title_text="Educação dos pais vs média de G1", height=400)
fig.show()

## 12. Conclusões

| Achado | Detalhe |
|---|---|
| **Distribuição de G1** | Aproximadamente normal, centrada ~10.7; existem alunos com nota 0 (possível abandono/doença) |
| **Maior preditor negativo** | `failures` — cada reprovação anterior está associada a quedas significativas na nota |
| **Maior preditor positivo** | `Medu` / educação da mãe — filhos de mães com ensino superior têm notas ~2 pontos acima |
| **Álcool** | `Dalc` e `Walc` correlacionam-se negativamente com G1 e positivamente entre si; também associados a mais faltas |
| **Faltas** | Distribuição muito assimétrica (skewed right) — a maioria tem poucas faltas, mas outliers chegam a 93 |
| **`higher`** | Alunos que querem prosseguir estudos têm mediana G1 ~2 pontos acima dos restantes |
| **Correlações fortes** | `Dalc`–`Walc` (r≈0.65), `Medu`–`Fedu` (r≈0.62), `G1`–`failures` (r≈-0.36) |
| **Dataset completo** | Sem valores em falta — ideal como ponto de partida para injecção de missingness artificial |